# 84 · Nhánh C seed 46 — một notebook ba phần (Tuấn duyệt 29/10)

**Kế hoạch:** `docs/PLAN_SAU_78Y.md` §A.4v, `docs/SPEC_SAU_78Y.md` §B.2ab, `Event.md` 2026-10-29 "Tuấn giữ nhánh C …". Mã: gói `nhanh_c` trên GitHub
(nhánh `nhanh-c`, commit ghim `2da38a76e91e`); mỗi ô gọi `nhanh_c.buoc(...)` — mã bước = ô đã kiểm của lượt gói ô cắt s42 (Phần 1) / lượt skip7 (Phần 2–3,
đường skip + 4 đầu chuẩn), khác đúng các dòng "nhanh C" theo seed.

1. **Phần 1 — `78x`** (`r84x_C_s46`): công thức lượt gói s42 (B.4 v2 + 5 ô lưới phóng 518 không gradient + adapter + skip block 7 → 8 + đầu J đọc 6 view);
   seed 46; khởi động luồng + λ chép J v3b s42. Cuối phần: **cổng sức khoẻ** (VAL trong lượt TB ep1–6 không thấp hơn C s42 0,2619 quá 0,01).
2. **Phần 2 — `78z` chuẩn** (`r84z_C_s46`): 4 đầu chuẩn trên backbone đọc ở **đỉnh VAL trong lượt** (bằng nhau → epoch muộn hơn); không dự đoán test mỗi epoch.
3. **Phần 3 — gộp trọng số** ep1–10 (`r84z_C_s46_gts`; đầu quyết định `fus_attn_hr`, luật mức 2 `C6S`) + **tổng hợp các seed đã xong** (TB ± độ lệch, bộ gộp
   các seed, C − A ở s42). Không đọc nhãn test.

**Chạy:** GPU **G4**, Restart session → **Run all**, ~21 giờ (ước: phần 1 ~12, phần 2 ~8, phần 3 ~1). Ngắt giữa chừng: Restart → Run all (phần 1 / 2 chạy
tiếp từ tệp resume). **Đừng tắt runtime trước `Phần 3 · C9`.** Ghi `stage4_nb84/…`.

**Gửi Claude:** ô tải gói; các dòng epoch + `O CAT` của `Phần 1 · C6`; dòng `CONG SUC KHOE` (`Phần 1 · C9`); dòng `EP_BB` (`Phần 2 · C0`); `Phần 2 · C2b`
+ `C7`; `Phần 3 · C6S` + toàn bộ `TỔNG HỢP`.

## Tải gói `nhanh_c` (GitHub, commit ghim) — chạy trước mọi ô


In [ ]:
# ==== TAI GOI nhanh_c tu GitHub (nhanh mo coi nhanh-c, GHIM commit) - chay MOI phien, TRUOC moi o khac ====
import os, subprocess, sys
NHANH_C_SHA = '2da38a76e91e64335cbde83dd82e507c388e16dd'
os.environ['CXRLT_GOC_ID'] = '1PqpiEocTXJNspAK4GM91bL7JoEaL1Bh_'   # ID thu muc cxrlt_2024 - CHI trong notebook (goi cong khai tren GitHub KHONG chua)
_d_goi = '/content/nhanh_c_goi'
_tep_goi = os.path.normcase(os.path.realpath(os.path.join(_d_goi, 'nhanh_c', '__init__.py')))
_goi_cu = sys.modules.get('nhanh_c')
if _goi_cu is not None:
    if os.path.normcase(os.path.realpath(getattr(_goi_cu, '__file__', ''))) != _tep_goi:
        raise RuntimeError('nhanh_c da nap tu duong dan khac -> Restart session truoc khi tai goi')
    if getattr(_goi_cu, '_NHANH_C_SHA_DA_NAP', None) != NHANH_C_SHA:
        raise RuntimeError('nhanh_c da nap tu commit khac hoac chua duoc loader ghim -> Restart session truoc khi tai goi')
if not os.path.isdir(os.path.join(_d_goi, '.git')):
    subprocess.run(['git', 'clone', '--quiet', '--branch', 'nhanh-c', 'https://github.com/AIVIETNAM-AIO-Tuan/TransferLearning-GPUtask.git', _d_goi], check=True)
subprocess.run(['git', '-C', _d_goi, 'fetch', '--quiet', 'origin', 'nhanh-c'], check=True)
subprocess.run(['git', '-C', _d_goi, 'checkout', '--quiet', '--detach', NHANH_C_SHA], check=True)
_head_goi = subprocess.run(['git', '-C', _d_goi, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert _head_goi == NHANH_C_SHA, f'goi nhanh_c: HEAD {_head_goi} KHAC commit ghim {NHANH_C_SHA} -> DUNG'
if _d_goi not in sys.path: sys.path.insert(0, _d_goi)
import nhanh_c as NC
if os.path.normcase(os.path.realpath(getattr(NC, '__file__', ''))) != _tep_goi:
    raise RuntimeError('nhanh_c import tu duong dan khac goi vua tai -> Restart session')
assert NC.PHIEN_BAN == 'nhanh_c_v1', NC.PHIEN_BAN
NC._NHANH_C_SHA_DA_NAP = NHANH_C_SHA
print(f'nhanh_c {NC.PHIEN_BAN} @ {NHANH_C_SHA[:12]} | {len(NC.ds_buoc())} buoc | seed 46' + (' (BAN THU)' if False else ''))
del _d_goi, _head_goi, _tep_goi, _goi_cu


# Phần 1 · `78x` nhánh C — `r84x_C_s<seed>` (6 epoch)

## Phần 1 · C0 · Cấu hình + dữ liệu + khối

Như J v2; thêm cấu hình `jca_v3.json` và khối `khoi_jca_v3` nhúng nguyên văn. J v3b: thêm cấu hình `jca_v3b.json` và khối `khoi_jca_v3b` (trần write-back) nhúng nguyên văn. Đợt 4: khối ghi Drive một lần (`khoi_ghi_cuoi`); cấu hình `jca_v3b_d4.json` (+ khối `khoi_on_dinh_ft` ở B.4) nhúng nguyên văn; lượt của notebook `BIEN_THE_D4`.

Ô cắt: cấu hình `O_CFG` (5 ô lưới phóng từ ảnh 518, khúc, adapter r 64, skip block 7 → 8, ngưỡng sàng lọc) + đoạn nguyên văn `khoi_cat_vung` / `khoi_cat_sau` + khối `khoi_fusion_study`, `khoi_cat_j`. Đầu ô: chặn rác output tại gốc (httplib2, HF, cảnh báo `requires_grad`; không đổi kết quả); AP mức study bằng `ap_cols` (nhãn không có study dương = NaN thay vì 0).

Nhánh C: cấu hình theo seed (`CAU_HINH_C` do `nhanh_c.buoc` đặt): `SEED`, `RUN_TAG` `r84x_C_s<seed>`, thư mục `stage4_nb84`; khởi động luồng + λ vẫn chép J v3b s42 (như lượt gói s42).


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C0', globals(), seed=46)


## Phần 1 · C1 · Kho ảnh 518 cục bộ (nguyên `78u`)

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C1', globals(), seed=46)


## Phần 1 · C2 · RAD-DINO + biến đổi + dataset + đầu đọc `38a` (nguyên `78u`) + `fwd_luong` (79a) + `fwd_luong_v2` / `dau_v2` / `loader_study` (78x) — J v3: `dau_v2` thêm đầu nguồn (logsumexp) và kênh vùng cố định

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C2', globals(), seed=46)


## Phần 1 · C3 · Bảng lúc train (`78u` + `79a`) + dữ liệu J v2

Neo theo view, ngữ nghĩa vùng của study (ý A), tập vùng S → target ý 2 (cây cha–con), 31 nhãn MIL. In độ phủ vị trí theo tier.

J v3: vùng cố định làm target dự phòng (14 nhãn), loss nhãn-vùng 33 nhãn; λ ×3,23; bảng nguồn nhãn từ `ngu_canh_train` (`78s`, chỉ train).

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C3', globals(), seed=46)


## Phần 1 · C4 · Module loss phụ + luồng vùng (neo theo mẫu) + DOCV / DAUV + cổng

Cổng: `fwd_luong_v2 = fwd_ft` lúc khởi tạo (write-back 0); `dau_h_v2` không token vùng = `dau_h`; MIL `w = 0` → logit = đầu đọc ý 1; lô kiểm có cặp frontal–lateral cùng study → mọi phần loss có mặt.

J v3: đầu nguồn + kênh vùng cố định (khởi tạo 0); write-back × 5; CE nguồn chỉ vào đầu nguồn. J v3b: gắn trần write-back theo ảnh (`ap_tran_ghi`) **sau** ×5. Đợt 3 (nhánh khe): tạo nhánh khe (hạt riêng), `ap_khe` sau trần; loss khe vào `PHAN_CFG`.

Ô cắt: module ô (adapter ×2, phép chiếu skip, bộ gộp 6 view) + `fwd_o` / `dau_v2_o`. Cổng: skip lúc đầu → đường ảnh toàn cảnh như B.4 v2; adapter = đồng nhất; 6 view giống nhau = đầu đọc gốc; trường nhìn ô; dòng ngẫu nhiên không đổi; dự đoán tất định (hai lần trùng từng bit — cổng đường chấm `C6` so từng bit); gộp 6 view dạng dày = `GopStudy`.


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C4', globals(), seed=46)


## Phần 1 · C4a · Lọc nhiễu khi tắt worker DataLoader (Python 3.13) — chạy trước `C4b`

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C4A', globals(), seed=46)


## Phần 1 · C4b · Khởi động luồng vùng (+ head nhãn-vùng)

Backbone, đầu đọc, tháp đông cứng; chỉ luồng + `PHU_V` + head nhãn-vùng học (1 epoch). Ý B không ở đây (cần logit đầu đọc).

Đợt 3: **chép** `luong_khoi_dong.pt` của J v3b s42; nhánh khe: **chép** `khe_khoi_dong_<nhánh>.pt` của notebook chuẩn bị.

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C4B', globals(), seed=46)


## Phần 1 · C5 · Tệp resume (nếu có) + λ **chép** từ J v3b s42

Không hiệu chỉnh lại; kiểm λ đang dùng trùng s42 từng số.

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C5', globals(), seed=46)


## Phần 1 · C6 · Huấn luyện 6 epoch — ghi Drive một lần

Mỗi epoch: `p_`, nhật ký, chẩn đoán giữ riêng, `rs_` nhẹ, tệp resume ghi đè; `backbone_ft.pt` (epoch 6) thêm `cfg_d4`. B.4: EMA sau mỗi bước, VAL EMA + thô.

Ô cắt: train / dự đoán / chẩn đoán giữ riêng qua `fwd_o` + `dau_v2_o`; EMA thêm 4 module ô; `rs_` / `backbone_ft.pt` / tệp resume thêm module ô + `cfg_o`; dòng `O CAT` mỗi epoch.

Nhánh C: cổng ghép cặp dấu vân lô chỉ ở s42; luật thử seed 44 của J v3b tắt.


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C6', globals(), seed=46)


## Phần 1 · C7 · Bảng lượt này cạnh J v3b s42 + dấu vân lô + λ + write-back / trần — báo kèm

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C7', globals(), seed=46)


## Phần 1 · C9 · `dong_bo()` (ghi một lần) — chạy trước khi tắt runtime; kiểm `rs_` nhẹ ep1–6 + `backbone_ft.pt`, tệp resume đã xoá

Nhánh C: **cổng sức khoẻ** — VAL trong lượt TB ep1–6 thấp hơn C s42 (0,2619) quá 0,01 → `YEU`, ô chuyển phần dừng (báo Claude); Tuấn quyết chạy tiếp: chạy ô `TIEP_TUC_C = True` rồi chạy lại ô chuyển.


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C9', globals(), seed=46)


## Chuyển phần 1 → 2 (cổng sức khoẻ nhánh C)


In [ ]:
import nhanh_c as NC; NC.buoc('chuyen_1_2', globals(), seed=46)


# Phần 2 · `78z` chuẩn trên backbone nhánh C — `r84z_C_s<seed>`

## Phần 2 · C0 · Config

In [ ]:
import nhanh_c as NC; NC.buoc('p2_C0', globals(), seed=46)


## Phần 2 · C1 · Không EHR / XGB / túi vi khuẩn / phim trước (luật chỉ-ảnh)

Thay `C1` của `53` / `56`: mọi đầu chỉ đọc ảnh, nên không nạp 217 cột EHR, 40 cột XGB, túi vi khuẩn, phim trước, và không tải tệp XGB test. Giữ chỗ các biến cùng tên (0 cột / rỗng / `PREV_ROW = -1`) để máy `78v` chạy nguyên văn.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C1', globals(), seed=46)


## Phần 2 · C2 · RAD-DINO + backbone `78x` (rs_ đỉnh VAL trong lượt) + biến đổi + DS

Nguyên văn `C2` của `78t` (nạp `NAP_BB_78G`, cổng tách đôi forward), thêm **cổng checkpoint `78x`**: thư mục / lượt / epoch của `rs_` phải đúng (và có `luong` với J) — sai → dừng.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C2', globals(), seed=46)


## Phần 2 · C2b · Luồng vùng J v3 đông cứng (chỉ lượt J) + 4 cổng

Dựng `LuongVungNeo` đúng cấu hình lưu trong `backbone_ft.pt` cùng lượt, write-back × g, nạp trọng số `rs_` ep4 (strict). Cổng: cấu hình đúng lượt; nạp strict; đường chia đôi (block 0–7 rồi block 8–11 + luồng) = forward đầy đủ; tỉ lệ write-back cùng cỡ nhật ký ep4 của `78x` (sai g → trượt). Neo theo view từ `view_group` (có sẵn lúc chấm). Lượt `B1_*`: `LUONG = None`.

**Seed 43 / 44 và dự đoán lại:** cờ neo của ảnh val / test lấy từ tệp view lúc chấm (CSV + `28e`); ảnh train giữ tệp cũ; cổng số ảnh đổi cờ = val 37 / test 114.

**J v3b (seed 42 / 43 / 44):** nhúng thêm khối `khoi_jca_v3b` và gắn trần write-back theo ảnh (ρ từ `cfg_jv3b` của `backbone_ft.pt`; thiếu `cfg_jv3b` → dừng) sau ×5.


Nhánh C: `C0` chọn `EP_BB` = đỉnh VAL EMA trong lượt (bằng nhau → epoch muộn hơn); `C2b` nạp `dua_o` + `bo_o7` đông cứng + cổng cấu trúc nhánh C + cổng 3′; 4 đầu chuẩn (đọc 1 view, đường skip).


In [ ]:
import nhanh_c as NC; NC.buoc('z_C2B', globals(), seed=46)


## Phần 2 · C2c · Study + view lúc chấm cho fusion + bảng đếm

`ST_FUS_T` (study của mỗi hàng ORDER), `VIEW_FUS_T` (frontal / lateral / khác, từ tệp view lúc chấm), generator riêng cho mặt nạ bỏ view. In: số ảnh / study, tỉ lệ study nhiều ảnh, phân bố view, và tỉ lệ study train có nhãn **khác nhau** giữa các ảnh (chỉ đếm — loss tính trên từng ảnh với logit chung của study).


In [ ]:
import nhanh_c as NC; NC.buoc('z_C2C', globals(), seed=46)


## Phần 2 · C3 · Kiến trúc — verbatim `53`/`56`

In [ ]:
import nhanh_c as NC; NC.buoc('z_c3', globals(), seed=46)


## Phần 2 · C3b · Hàm mất mát — `ASL(4,0)` như `64` (mặc định) hoặc `wasl25` (`LOSS_78D`)


In [ ]:
import nhanh_c as NC; NC.buoc('z_C3B', globals(), seed=46)


## Phần 2 · C3c · Khối dùng chung + bảng mức ảnh / study từ `78e` / `78f` / `78q` / `78s` (chỉ train)

Chép nguyên văn bảy khối trong `experiments/78_chi_anh/khoi/` (thêm `khoi_cau`, `khoi_txtq`, `khoi_dich`) và định nghĩa `txtq` của `78g`. Bảng cho loss phụ chỉ gồm ảnh / study của bệnh nhân train, trừ 2% bệnh nhân giữ riêng; câu report chỉ dùng theo **mã câu** (không đọc văn bản). Kiểm hợp đồng dữ liệu `78s` (kiểu, hình, giá trị, nhãn dương, thứ tự ảnh train, mô hình BiomedBERT ghim). Cổng hình học như `78g`. Không ghi gì lên Drive.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C3C', globals(), seed=46)


## Phần 2 · C4 · Bốn đầu (`mldec`, `mldec_hr`, `fus_tb_hr`, `fus_attn_hr`) — máy `C4` của `78t` với `ARMS_GIU`

`fus_*_hr` = ML-Decoder + `DocVung` khởi tạo như `mldec_hr` + `GopStudy` (tham số 0). Cổng khởi tạo như `78t` / `78y` + **cổng fusion J**: study một ảnh → logit fusion = `mldec_hr` từng bit; lúc đầu `fus_tb_hr` = `fus_attn_hr`; study hai ảnh = trung bình logit.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C4', globals(), seed=46)


## Phần 2 · C5 · Chấm điểm

In [ ]:
import nhanh_c as NC; NC.buoc('z_c5', globals(), seed=46)


## Phần 2 · C5b · Lọc nhiễu khi tắt worker DataLoader (Python 3.13)

Ẩn đúng ba thông báo vô hại sinh ra khi tắt worker (`QueueFeederThread` Bad file descriptor, `_ConnectionBase.__del__`, "can only test a child process"); mọi lỗi khác vẫn in. Không đổi dữ liệu / kết quả. `print(_NHIEU_78V)`: số thông báo đã ẩn trong tiến trình chính.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C5B', globals(), seed=46)


## Phần 2 · C6 · Huấn luyện bốn đầu — `C6` của `78t` + đường J (`78y`) + đường fusion (`78z`) + ghi một lần

`upper_frozen` chạy luồng vùng đông cứng (neo theo view) và gắn hàng của lô; đầu fusion gộp đầu ra query của `mldec_hr` theo `study_id` + view. Lô train theo study, lô dự đoán gói trọn study. Cổng đường chấm + cổng fusion (lô dự đoán không cắt study, logit một study không phụ thuộc study khác, eval bỏ qua mặt nạ). Tệp kết quả chỉ ghi local; mỗi epoch ghi đè **một** tệp resume; cuối cell `dong_bo()` tải tất cả lên một lượt.


In [ ]:
import nhanh_c as NC; NC.buoc('p2_C6', globals(), seed=46)


## Phần 2 · C7 · Bảng kết quả từng epoch (chạy sau `C6`: tệp lên Drive ở cuối `C6`)

Theo dõi, chưa kết luận: đầu fusion đã gộp theo study, các đầu khác chưa gộp view. Số quyết định (`fus_attn_hr − mldec_hr`, trung bình 10 epoch, gộp view, hai nửa val) ở bảng `78a` của bản này.


In [ ]:
import nhanh_c as NC; NC.buoc('p2_C7', globals(), seed=46)


## Phần 2 · C9 · `dong_bo()` — chạy trước khi tắt runtime (+ danh sách `rs_` có thể xoá: Tuấn quyết, không tự xoá)


In [ ]:
import nhanh_c as NC; NC.buoc('p2_C9', globals(), seed=46)


## Chuyển phần 2 → 3


In [ ]:
import nhanh_c as NC; NC.buoc('chuyen_2_3', globals(), seed=46)


# Phần 3 · Gộp trọng số đầu đọc ep1–10 + tổng hợp các seed — `r84z_C_s<seed>_gts`

## Phần 3 · C0 · Config

In [ ]:
import nhanh_c as NC; NC.buoc('p3_C0', globals(), seed=46)


## Phần 3 · C1 · Không EHR / XGB / túi vi khuẩn / phim trước (luật chỉ-ảnh)

Thay `C1` của `53` / `56`: mọi đầu chỉ đọc ảnh, nên không nạp 217 cột EHR, 40 cột XGB, túi vi khuẩn, phim trước, và không tải tệp XGB test. Giữ chỗ các biến cùng tên (0 cột / rỗng / `PREV_ROW = -1`) để máy `78v` chạy nguyên văn.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C1', globals(), seed=46)


## Phần 3 · C2 · RAD-DINO + backbone `78x` (rs_ đỉnh VAL trong lượt) + biến đổi + DS

Nguyên văn `C2` của `78t` (nạp `NAP_BB_78G`, cổng tách đôi forward), thêm **cổng checkpoint `78x`**: thư mục / lượt / epoch của `rs_` phải đúng (và có `luong` với J) — sai → dừng.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C2', globals(), seed=46)


## Phần 3 · C2b · Luồng vùng J v3 đông cứng (chỉ lượt J) + 4 cổng

Dựng `LuongVungNeo` đúng cấu hình lưu trong `backbone_ft.pt` cùng lượt, write-back × g, nạp trọng số `rs_` ep4 (strict). Cổng: cấu hình đúng lượt; nạp strict; đường chia đôi (block 0–7 rồi block 8–11 + luồng) = forward đầy đủ; tỉ lệ write-back cùng cỡ nhật ký ep4 của `78x` (sai g → trượt). Neo theo view từ `view_group` (có sẵn lúc chấm). Lượt `B1_*`: `LUONG = None`.

**Seed 43 / 44 và dự đoán lại:** cờ neo của ảnh val / test lấy từ tệp view lúc chấm (CSV + `28e`); ảnh train giữ tệp cũ; cổng số ảnh đổi cờ = val 37 / test 114.

**J v3b (seed 42 / 43 / 44):** nhúng thêm khối `khoi_jca_v3b` và gắn trần write-back theo ảnh (ρ từ `cfg_jv3b` của `backbone_ft.pt`; thiếu `cfg_jv3b` → dừng) sau ×5.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C2B', globals(), seed=46)


## Phần 3 · C2c · Study + view lúc chấm cho fusion + bảng đếm

`ST_FUS_T` (study của mỗi hàng ORDER), `VIEW_FUS_T` (frontal / lateral / khác, từ tệp view lúc chấm), generator riêng cho mặt nạ bỏ view. In: số ảnh / study, tỉ lệ study nhiều ảnh, phân bố view, và tỉ lệ study train có nhãn **khác nhau** giữa các ảnh (chỉ đếm — loss tính trên từng ảnh với logit chung của study).


In [ ]:
import nhanh_c as NC; NC.buoc('z_C2C', globals(), seed=46)


## Phần 3 · C3 · Kiến trúc — verbatim `53`/`56`

In [ ]:
import nhanh_c as NC; NC.buoc('z_c3', globals(), seed=46)


## Phần 3 · C3b · Hàm mất mát — `ASL(4,0)` như `64` (mặc định) hoặc `wasl25` (`LOSS_78D`)


In [ ]:
import nhanh_c as NC; NC.buoc('z_C3B', globals(), seed=46)


## Phần 3 · C3c · Khối dùng chung + bảng mức ảnh / study từ `78e` / `78f` / `78q` / `78s` (chỉ train)

Chép nguyên văn bảy khối trong `experiments/78_chi_anh/khoi/` (thêm `khoi_cau`, `khoi_txtq`, `khoi_dich`) và định nghĩa `txtq` của `78g`. Bảng cho loss phụ chỉ gồm ảnh / study của bệnh nhân train, trừ 2% bệnh nhân giữ riêng; câu report chỉ dùng theo **mã câu** (không đọc văn bản). Kiểm hợp đồng dữ liệu `78s` (kiểu, hình, giá trị, nhãn dương, thứ tự ảnh train, mô hình BiomedBERT ghim). Cổng hình học như `78g`. Không ghi gì lên Drive.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C3C', globals(), seed=46)


## Phần 3 · C4 · Bốn đầu (`mldec`, `mldec_hr`, `fus_tb_hr`, `fus_attn_hr`) — máy `C4` của `78t` với `ARMS_GIU`

`fus_*_hr` = ML-Decoder + `DocVung` khởi tạo như `mldec_hr` + `GopStudy` (tham số 0). Cổng khởi tạo như `78t` / `78y` + **cổng fusion J**: study một ảnh → logit fusion = `mldec_hr` từng bit; lúc đầu `fus_tb_hr` = `fus_attn_hr`; study hai ảnh = trung bình logit.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C4', globals(), seed=46)


## Phần 3 · C5 · Chấm điểm

In [ ]:
import nhanh_c as NC; NC.buoc('z_c5', globals(), seed=46)


## Phần 3 · C5b · Lọc nhiễu khi tắt worker DataLoader (Python 3.13)

Ẩn đúng ba thông báo vô hại sinh ra khi tắt worker (`QueueFeederThread` Bad file descriptor, `_ConnectionBase.__del__`, "can only test a child process"); mọi lỗi khác vẫn in. Không đổi dữ liệu / kết quả. `print(_NHIEU_78V)`: số thông báo đã ẩn trong tiến trình chính.


In [ ]:
import nhanh_c as NC; NC.buoc('z_C5B', globals(), seed=46)


## Phần 3 · C6S · Gộp trọng số: nạp `w_` / `p_` ep1–10, cổng tái lập, hai bộ gộp, luật đặt trước, ghi tệp

Phần định nghĩa + cổng của `C6` (lượt fusion J) nguyên văn, rồi phần mới ở cuối cell. Không train, không lambda, không resume, không đọc nhãn test.


In [ ]:
import nhanh_c as NC; NC.buoc('p3_C6S', globals(), seed=46)


## Phần 3 · TỔNG HỢP · Các seed nhánh C đã xong — `fus_attn_hr` gộp trọng số + TB dự đoán, TB ± độ lệch, bộ gộp các seed (chạy trước `C9`)


In [ ]:
import nhanh_c as NC; NC.buoc('p3_TONG_HOP', globals(), seed=46)


## Phần 3 · C9 · `dong_bo()` — chạy trước khi tắt runtime


In [ ]:
import nhanh_c as NC; NC.buoc('p3_C9S', globals(), seed=46)
